In [1]:
import time
import requests
import pandas as pd
from collections import Counter

In [ ]:
# =========================
# CONFIG
# =========================
API_KEY = "aNPobAtZIsrciuiwkj12ov"   # 建议填：OpenAlex api_key
MAILTO = "daisyfire0720@hotmail.com"    # 建议填：你的邮箱
TARGET_DOI = "10.1016/j.apenergy.2022.119295"

BASE = "https://api.openalex.org"

# =========================
# Session (more stable)
# =========================
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (citation-map; contact=mailto)"  # 简单 UA
})

def _get_json(url, params=None, max_retries=8, base_sleep=0.6):
    """
    Robust GET that:
    - Retries on 429/5xx
    - Retries when response is not JSON (HTML/empty)
    - Prints minimal diagnostics on repeated failures
    """
    if not url:
        raise ValueError("URL is empty/None")

    params = params.copy() if params else {}
    if API_KEY:
        params["api_key"] = API_KEY
    if MAILTO:
        params["mailto"] = MAILTO

    last_status = None
    last_text_head = None

    for i in range(max_retries):
        r = session.get(url, params=params, timeout=45)
        last_status = r.status_code
        text = r.text or ""
        last_text_head = text[:200].replace("\n", " ")

        # success path: status 200 + json
        if r.status_code == 200:
            ctype = (r.headers.get("Content-Type") or "").lower()
            # OpenAlex 正常是 application/json
            if "json" in ctype and text.strip():
                try:
                    return r.json()
                except Exception:
                    # 200 but not parseable json -> retry
                    pass
            # 200 but HTML or empty -> retry
        elif r.status_code in (429, 500, 502, 503, 504):
            pass
        else:
            raise RuntimeError(
                f"Request failed: {r.status_code}\nURL: {r.url}\nBody head: {last_text_head}"
            )

        # backoff
        sleep_s = (2 ** i) * base_sleep
        time.sleep(sleep_s)

    raise RuntimeError(
        f"Failed after retries. Last status={last_status}\nURL: {url}\nBody head: {last_text_head}"
    )

def to_api_work_url(openalex_id: str) -> str:
    """
    Convert OpenAlex work ID to API endpoint.
    Examples:
      https://openalex.org/W123  -> https://api.openalex.org/works/W123
      W123                      -> https://api.openalex.org/works/W123
    """
    if not openalex_id:
        raise ValueError("Empty OpenAlex ID")

    oid = openalex_id.strip()
    if oid.startswith("https://openalex.org/"):
        oid = oid.replace("https://openalex.org/", "")
    if oid.startswith("http://openalex.org/"):
        oid = oid.replace("http://openalex.org/", "")

    # oid should now look like Wxxxx...
    return f"{BASE}/works/{oid}"

# =========================
# 1) Get target work by DOI
# =========================
work_url = f"{BASE}/works/https://doi.org/{TARGET_DOI}"
work = _get_json(work_url)

target_openalex_id = work.get("id")
target_title = work.get("display_name")
target_year = work.get("publication_year")

print("Target title:", target_title)
print("Target year :", target_year)
print("Target OAID :", target_openalex_id)

if not target_openalex_id:
    raise RuntimeError("Failed to get target_openalex_id. Check DOI correctness.")

# =========================
# 2) Get all citing works using cursor paging (stable)
# filter=cites:<target_openalex_id>
# =========================
per_page = 200
cursor = "*"   # cursor paging start
citing_rows = []

while True:
    url = f"{BASE}/works"
    params = {
        "filter": f"cites:{target_openalex_id}",
        "per-page": per_page,
        "cursor": cursor,
        # 只取我们需要的字段，降低 payload（更快、更少触发限流）
        "select": "id,display_name,publication_year,doi,primary_location,cited_by_count"
    }
    data = _get_json(url, params=params)
    results = data.get("results", [])
    if not results:
        break

    for w in results:
        citing_rows.append({
            "citing_openalex_id": w.get("id"),
            "citing_title": w.get("display_name"),
            "citing_year": w.get("publication_year"),
            "citing_doi": (w.get("doi") or "").replace("https://doi.org/", ""),
            "citing_venue": (w.get("primary_location") or {}).get("source", {}).get("display_name"),
            "cited_by_count": w.get("cited_by_count"),
        })

    cursor = data.get("meta", {}).get("next_cursor")
    if not cursor:
        break

    time.sleep(0.12)

citing_df = pd.DataFrame(citing_rows).drop_duplicates(subset=["citing_openalex_id"]).reset_index(drop=True)
print("Total citing works:", len(citing_df))

# =========================
# 3) Extract country codes (paper-level unique countries)
# To reduce API load, fetch only authorships+institutions fields
# =========================
# =========================
# 3) Extract country codes (paper-level unique countries)
# =========================
paper_to_countries = {}
paper_country_list = []

for idx, row in citing_df.iterrows():
    wid = row["citing_openalex_id"]
    if not wid:
        continue

    # IMPORTANT: convert to API endpoint
    wobj = _get_json(
    to_api_work_url(wid),
    params={"select": "id,authorships"}
)

    countries = set()
    for auth in (wobj.get("authorships") or []):
        for inst in (auth.get("institutions") or []):
            cc = inst.get("country_code")
            if cc:
                countries.add(cc)

    cc_sorted = sorted(countries)
    paper_to_countries[wid] = cc_sorted

    for cc in cc_sorted:
        paper_country_list.append(cc)

    if (idx + 1) % 20 == 0:
        print(f"Processed countries for {idx+1}/{len(citing_df)} citing papers...")

    time.sleep(0.05)

# Attach country list back
citing_df["country_codes"] = citing_df["citing_openalex_id"].map(lambda x: ";".join(paper_to_countries.get(x, [])))

# =========================
# 4) Country counts
# =========================
country_counts = Counter(paper_country_list)
country_df = (
    pd.DataFrame(country_counts.items(), columns=["country_code", "num_citing_papers"])
      .sort_values("num_citing_papers", ascending=False)
      .reset_index(drop=True)
)

print("\nTop countries:")
print(country_df.head(10))

# =========================
# 5) Gephi edges + nodes
# =========================
edges_df = pd.DataFrame({
    "source": [target_openalex_id] * len(citing_df),
    "target": citing_df["citing_openalex_id"].tolist(),
    "type": ["Directed"] * len(citing_df)
})

nodes = [{"id": target_openalex_id, "label": target_title, "year": target_year, "country_codes": ""}]
for _, r in citing_df.iterrows():
    nodes.append({
        "id": r["citing_openalex_id"],
        "label": r["citing_title"],
        "year": r["citing_year"],
        "country_codes": r["country_codes"]
    })
nodes_df = pd.DataFrame(nodes).drop_duplicates(subset=["id"]).reset_index(drop=True)

# =========================
# 6) Save outputs (pre-plot)
# =========================
citing_df.to_csv("citing_works.csv", index=False)
country_df.to_csv("country_counts.csv", index=False)
edges_df.to_csv("edges.csv", index=False)
nodes_df.to_csv("nodes.csv", index=False)

print("\nSaved files:")
print(" - citing_works.csv")
print(" - country_counts.csv")
print(" - edges.csv")
print(" - nodes.csv")


Target title: Electric vehicle market potential and associated energy and emissions reduction benefits
Target year : 2022
Target OAID : https://openalex.org/W4282983575
Total citing works: 30
Processed countries for 20/30 citing papers...

Top countries:
  country_code  num_citing_papers
0           CN                 14
1           US                  7
2           MY                  4
3           IN                  4
4           GB                  3
5           AE                  2
6           JP                  2
7           IE                  1
8           TR                  1
9           SA                  1

Saved files:
 - citing_works.csv
 - country_counts.csv
 - edges.csv
 - nodes.csv


In [3]:
# =========================
# PLOTTING (Plotly) - Full code
# Inputs:
#   - country_counts.csv  (columns: country_code, num_citing_papers)
# Outputs:
#   - citation_country_map.html / .png
#   - citation_country_bar.html / .png
# =========================

import pandas as pd
import plotly.express as px

# ---- 1) Load data ----
country_df = pd.read_csv("country_counts.csv")

# Basic sanity check
required_cols = {"country_code", "num_citing_papers"}
missing = required_cols - set(country_df.columns)
if missing:
    raise ValueError(f"country_counts.csv missing columns: {missing}")

country_df["country_code"] = country_df["country_code"].astype(str).str.upper().str.strip()
country_df["num_citing_papers"] = pd.to_numeric(country_df["num_citing_papers"], errors="coerce").fillna(0).astype(int)

# ---- 2) ISO2 -> ISO3 conversion ----
# Plotly choropleth works best with ISO-3 codes.
try:
    import pycountry
except ImportError:
    raise ImportError("Please install pycountry: pip install -U pycountry")

def iso2_to_iso3(iso2: str):
    iso2 = (iso2 or "").strip().upper()
    if not iso2:
        return None

    # Special cases (rare)
    special = {
        "XK": "XKX",  # Kosovo sometimes appears as XK
    }
    if iso2 in special:
        return special[iso2]

    c = pycountry.countries.get(alpha_2=iso2)
    return c.alpha_3 if c else None

plot_df = country_df.copy()
plot_df["iso3"] = plot_df["country_code"].apply(iso2_to_iso3)

# Drop rows that can't be mapped (usually very few)
plot_df = plot_df.dropna(subset=["iso3"]).reset_index(drop=True)

if plot_df.empty:
    raise RuntimeError("No countries could be mapped to ISO-3 codes. Check country_code values.")

# ---- 3) World map (Choropleth) ----
map_fig = px.choropleth(
    plot_df,
    locations="iso3",
    color="num_citing_papers",
    hover_name="country_code",
    hover_data={"num_citing_papers": True, "iso3": False},
    labels={"num_citing_papers": "Number of Citing Papers"},
    title="Geographic Distribution of Citing Papers (OpenAlex)"
)

map_fig.update_layout(
    margin=dict(l=20, r=20, t=70, b=20)
)

# Save interactive HTML
map_html = "citation_country_map.html"
map_fig.write_html(map_html)
print(f"Saved: {map_html}")

# Save PNG (optional, requires kaleido)
map_png = "citation_country_map.png"
try:
    map_fig.write_image(map_png, scale=2)
    print(f"Saved: {map_png}")
except Exception as e:
    print("PNG export skipped. Install kaleido to enable PNG export:")
    print("  pip install -U kaleido")
    print("Error:", e)

map_fig.show()


# ---- 4) Country bar chart (Top N) ----
TOP_N = 20
bar_df = plot_df.sort_values("num_citing_papers", ascending=False).head(TOP_N)

bar_fig = px.bar(
    bar_df,
    x="country_code",
    y="num_citing_papers",
    labels={"country_code": "Country", "num_citing_papers": "Number of Citing Papers"},
    title=f"Top {TOP_N} Countries by Number of Citing Papers (OpenAlex)"
)

bar_fig.update_layout(
    margin=dict(l=20, r=20, t=70, b=20),
    xaxis_tickangle=-45
)

bar_html = "citation_country_bar.html"
bar_fig.write_html(bar_html)
print(f"Saved: {bar_html}")

bar_png = "citation_country_bar.png"
try:
    bar_fig.write_image(bar_png, scale=2)
    print(f"Saved: {bar_png}")
except Exception as e:
    print("PNG export skipped. Install kaleido to enable PNG export:")
    print("  pip install -U kaleido")
    print("Error:", e)

bar_fig.show()


Saved: citation_country_map.html
Saved: citation_country_map.png


Saved: citation_country_bar.html
Saved: citation_country_bar.png
